In [ ]:
from datetime import datetime
import re
from IPython.display import display, HTML


In [ ]:

# SOURCE DATA 

CUSTOMERS = {
    "Priya Nair": {
        "loyalty_tier": "Gold",
        "pnr": "SK4821X",
        "contact": "priya.nair@example.com, +91-98xxxxxxx1",
        "travel_history": "6 flights, 1 prior complaint (delayed baggage, resolved with voucher)"
    },
    "Arvind Kulkarni": {
        "loyalty_tier": "Silver",
        "pnr": "TR1190B",
        "contact": "arvind.kulkarni@example.com, +91-98xxxxxxx2",
        "travel_history": "3 flights, no prior complaints"
    },
    "Meher Kaur": {
        "loyalty_tier": "Platinum",
        "pnr": "WL7742",
        "contact": "meher.kaur@example.com, +91-98xxxxxxx3",
        "travel_history": "10 flights, 1 prior complaint (overbooking, resolved with a tier-status upgrade)"
    }
}

BOOKINGS = {
    "Priya Nair": [
        {"pnr":"SK4821X","flight":"SK-204","route":"Delhi → Goa","date":"Wed 23 Sep 2026",
         "departure":"18:40","status":"Cancelled","detail":"operational reasons"},
        {"pnr":"SK4821X","flight":"Return","route":"Goa → Delhi","date":"Fri 25 Sep 2026",
         "departure":"16:20","status":"Unaffected","detail":""}
    ],
    "Arvind Kulkarni": [
        {"pnr":"TR1190B","flight":"SK-118","route":"Mumbai → Bengaluru","date":"Wed 23 Sep 2026",
         "departure":"07:10","status":"Delayed 4h","detail":"new departure 11:10"}
    ],
    "Meher Kaur": [
        {"pnr":"WL7742","flight":"SK-305","route":"Delhi → Hyderabad","date":"Wed 23 Sep 2026",
         "departure":"14:00","status":"Delayed 6h","detail":"new departure 20:00"}
    ]
}


In [ ]:

# PER-PERSONA SESSION STATE


SESSIONS = {
    customer: {
        "conversation": [],
        "action_log": [],
        "executed_actions": []
    }
    for customer in CUSTOMERS
}

def current_session(customer):
    return SESSIONS[customer]

def reset_persona(customer):
    SESSIONS[customer] = {
        "conversation": [],
        "action_log": [],
        "executed_actions": []
    }

def log_action(customer, action, status, reason):
    current_session(customer)["action_log"].append({
        "timestamp": datetime.now().strftime("%H:%M:%S"),
        "customer": customer,
        "action": action,
        "status": status,
        "reason": reason
    })


def primary_booking(customer):
    return BOOKINGS[customer][0]

def delay_hours(booking):
    status = booking.get("status", "")
    digits = re.search(r"(\d+)", status)
    return int(digits.group(1)) if digits else 0


In [ ]:

def detect_tone(message):
    text = message.lower()
    if any(x in text for x in [
        "furious", "angry", "unacceptable", "ridiculous", "terrible",
        "worst", "legal action", "lawyer", "sue"
    ]):
        return "frustrated"
    if any(x in text for x in ["confused", "don't understand", "what can you do", "help"]):
        return "confused"
    return "neutral"

def detect_intents(message):
    text = message.lower()
    intents = []

    if any(x in text for x in ["refund", "money back", "cash back"]):
        intents.append("refund")
    if any(x in text for x in ["rebook", "another flight", "move me", "different flight"]):
        intents.append("rebooking")
    if any(x in text for x in ["business class", "upgrade", "free upgrade"]):
        intents.append("upgrade")
    if any(x in text for x in ["meal", "food", "voucher"]):
        intents.append("meal_voucher")
    if "lounge" in text:
        intents.append("lounge")
    if any(x in text for x in ["hotel", "stay", "accommodation"]):
        intents.append("hotel")
    if any(x in text for x in ["fare difference", "₹2000", "2000", "higher fare", "higher-fare"]):
        intents.append("fare_difference")
    if any(x in text for x in ["legal", "lawyer", "sue", "lawsuit", "formal complaint"]):
        intents.append("legal_or_complaint")

    return intents


In [ ]:

def policy_decision(customer, intent):
    booking = primary_booking(customer)
    status = booking["status"].lower()
    delay = delay_hours(booking)

    if intent == "legal_or_complaint":
        return ("ESCALATE", "Human escalation",
                "Legal action or a formal complaint must be escalated immediately.")

    if intent == "refund":
        if "cancelled" in status and "operational" in booking["detail"].lower():
            return ("EXECUTE", "Initiate full refund",
                    "Airline-caused cancellation qualifies for a full refund to the original payment method.")
        return ("ESCALATE", "Refund review",
                "The supplied data does not authorize this refund request.")

    if intent == "rebooking":
        if "cancelled" in status and "operational" in booking["detail"].lower():
            return ("EXECUTE", "Free rebooking",
                    "Airline-caused cancellation qualifies for free rebooking on the next available flight within 24 hours.")
        return ("RECOMMEND", "No automatic rebooking",
                "The supplied data does not establish eligibility for this rebooking request.")

    if intent == "meal_voucher":
        if delay > 3:
            return ("EXECUTE", "Issue meal voucher", "Delay is more than 3 hours.")
        if delay > 0:
            return ("EXECUTE", "Issue ₹500 meal voucher", "Delay is under 3 hours.")

    if intent == "lounge":
        if delay > 3:
            return ("EXECUTE", "Provide lounge access", "Delay is more than 3 hours.")
        return ("RECOMMEND", "Lounge not covered",
                "The supplied policy does not grant lounge access for this delay.")

    if intent == "hotel":
        if delay > 5:
            return ("EXECUTE", "Arrange hotel for delayed hours",
                    "Delay is more than 5 hours; policy covers only the delayed-hours portion.")
        return ("RECOMMEND", "Hotel not covered",
                "Hotel accommodation applies only to delays of more than 5 hours.")

    if intent == "upgrade":
        return ("ESCALATE", "Human review for upgrade",
                "The supplied policies do not authorize a free business-class upgrade.")

    if intent == "fare_difference":
        return ("ESCALATE", "Supervisor approval",
                "A fare difference above ₹1,500 requires supervisor approval; this request is ₹2,000.")

    return ("RECOMMEND", "Clarification",
            "One necessary clarification is required before taking an action.")


In [ ]:

# SIMULATED EXECUTION + CONVERSATIONAL AGENT


def execute_action(customer, action):
    session = current_session(customer)

    results = {
        "Initiate full refund":
            "Full refund request initiated to the original payment method. Processing time: within 7 business days.",
        "Free rebooking":
            "Rebooking request executed for the next available flight within 24 hours at no charge.",
        "Issue meal voucher":
            "Meal voucher issued according to the delay policy.",
        "Issue ₹500 meal voucher":
            "₹500 meal voucher issued according to the delay policy.",
        "Provide lounge access":
            "Lounge access provisioned according to the delay policy.",
        "Arrange hotel for delayed hours":
            "Hotel accommodation arranged for the delayed-hours portion only; not a full-night stay."
    }

    result = results.get(action)
    if result:
        session["executed_actions"].append({
            "customer": customer,
            "action": action,
            "result": result
        })
        log_action(customer, action, "EXECUTED",
                   "Action is explicitly allowed by the supplied policy.")
        return result

    log_action(customer, action, "ESCALATED",
               "Action requires human authority.")
    return "This request has been escalated to a human agent."

def agent_reply(customer, message):
    session = current_session(customer)
    tone = detect_tone(message)
    intents = detect_intents(message)

    session["conversation"].append({"role": "customer", "text": message})

    if not intents:
        reply = (
            "I can help with your current flight disruption. "
            "Would you like a refund, rebooking, or help with the applicable delay benefits?"
        )
        session["conversation"].append({"role": "agent", "text": reply})
        return reply

    responses = []

    if tone == "frustrated":
        responses.append(
            "I understand this has been frustrating, and I’ll help with what I’m authorized to do."
        )

    for intent in intents:
        decision, action, reason = policy_decision(customer, intent)

        if decision == "EXECUTE":
            responses.append(execute_action(customer, action))

        elif decision == "ESCALATE":
            log_action(customer, action, "ESCALATED", reason)
            responses.append(
                "I can’t complete that request directly. " +
                reason + " I’m escalating it to a human agent."
            )

        else:
            responses.append(reason)

    booking = primary_booking(customer)

    if "cancelled" in booking["status"].lower() and not any(
        x in intents for x in ["refund", "rebooking"]
    ):
        responses.append(
            "Because this flight was cancelled for operational reasons, you may choose "
            "either free rebooking on the next available flight within 24 hours or a full refund."
        )

    reply = " ".join(responses)
    session["conversation"].append({"role": "agent", "text": reply})
    return reply


In [ ]:
# STANDALONE WEB UI



display(HTML(r"""
<style>
html,body,.jp-Notebook,.jp-NotebookPanel{background:#f5f2ed!important}
.jp-Cell{border:0!important;box-shadow:none!important}
.aionos-app{font-family:Arial,'DM Sans',sans-serif;color:#17222d;max-width:1500px;margin:auto;padding:18px 26px 34px}
.a-head{display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid #ddd6ce;padding:5px 2px 16px}
.a-brand{display:flex;align-items:center;gap:12px}.a-mark{color:#8d2435;font-size:28px}.a-name{font-family:Georgia,serif;color:#8d2435;font-size:27px}.a-divider{height:31px;border-left:1px solid #cfc6bc}.a-sub{font-family:Georgia,serif;letter-spacing:3px;font-size:13px}.a-caption{font-size:8px;letter-spacing:2px;color:#8a9096;margin-top:3px}
.a-note{font-family:Georgia,serif;font-style:italic;color:#777;font-size:13px}.a-note span{display:inline-block;width:55px;border-top:1px solid #b8aea3;margin:0 10px 4px}
.a-hero{display:grid;grid-template-columns:1.4fr .6fr;gap:15px;margin:16px 0 14px}.a-card{background:#fff;border:1px solid #ded8d0;border-radius:15px;box-shadow:0 8px 24px rgba(50,40,30,.045)}
.a-main{padding:25px 29px;position:relative;overflow:hidden}.a-main:after{content:"";position:absolute;width:240px;height:240px;border:1px solid #ead7d4;border-radius:50%;right:-70px;top:-85px;box-shadow:0 0 0 22px #fcf8f6,0 0 0 23px #eee2de,0 0 0 48px #fff,0 0 0 49px #f0ebe6}
.a-eyebrow{font-size:8.5px;font-weight:700;letter-spacing:1.7px;text-transform:uppercase;color:#777f87}.a-title{font:600 35px/1.04 Georgia,serif;margin:6px 0 9px;position:relative;z-index:1}.a-desc{font-size:11.5px;line-height:1.6;color:#68717d;max-width:750px;position:relative;z-index:1}
.a-mode{padding:25px;display:flex;flex-direction:column;justify-content:center}.a-mode strong{font-size:19px;margin-top:5px}.a-mode small{font-size:9.5px;color:#68717d;line-height:1.5;margin-top:6px}
.a-label{font-size:8.5px;font-weight:700;letter-spacing:1.6px;text-transform:uppercase;color:#777f87;margin:0 0 7px 2px}
.a-personas{display:flex;gap:8px;margin-bottom:15px}.persona{flex:0 0 180px;height:44px;border:1px solid #d8d1c8;border-radius:12px;background:#fff;color:#26313b;font-weight:600;cursor:pointer;box-shadow:0 4px 13px rgba(43,37,30,.035);transition:.18s}.persona:hover{border-color:#8d2435;transform:translateY(-1px);box-shadow:0 8px 18px rgba(43,37,30,.08)}.persona.active{background:#f5e6e8;border-color:#8d2435;color:#7a1e2d;box-shadow:0 5px 17px rgba(141,36,53,.10)}
.a-grid{display:grid;grid-template-columns:255px minmax(520px,1fr) 285px;gap:15px;align-items:start}.a-panel{background:#fff;border:1px solid #ded8d0;border-radius:15px;overflow:hidden;box-shadow:0 8px 24px rgba(50,40,30,.04)}.a-pt{padding:13px 16px;border-bottom:1px solid #ebe7e1;font-size:8.5px;font-weight:700;letter-spacing:1.15px;text-transform:uppercase}.a-pb{padding:14px 16px}
.info{padding:8px 0;border-bottom:1px solid #f0ede8}.info:last-child{border:0}.k{font-size:7.8px;color:#858c93;text-transform:uppercase;letter-spacing:.7px}.v{font-size:11.5px;font-weight:600;margin-top:3px}.status{display:inline-block;padding:4px 8px;border-radius:12px;background:#f5e5e7;color:#8d2435;font-size:7.5px;font-weight:700;margin-top:4px}
.policy{background:#faf5f2;border-left:3px solid #8d2435;padding:9px 10px;margin-bottom:8px;font-size:9.5px;line-height:1.42}
.chat-head{padding:13px 17px;border-bottom:1px solid #ebe7e1;display:flex;justify-content:space-between}.chat-name{font-size:12px;font-weight:700}.pnr{font-size:8.5px;color:#808891;margin-top:2px}.live{font-size:8px;color:#34735a;font-weight:700;letter-spacing:1px}.dot{display:inline-block;width:6px;height:6px;border-radius:50%;background:#3a8b69;margin-right:5px}
.chat{height:330px;overflow-y:auto;padding:18px 20px;background:#fbfaf8}.empty{height:100%;display:flex;align-items:center;justify-content:center;text-align:center;color:#707983;font-size:11px;line-height:1.65}.empty-icon{width:44px;height:44px;border:1px solid #d9d3cb;border-radius:50%;margin:0 auto 10px;display:flex;align-items:center;justify-content:center;color:#8d2435;font-size:18px}
.bubble{max-width:77%;padding:11px 13px;border-radius:11px;margin-bottom:10px;font-size:11.5px;line-height:1.52}.customer{margin-left:auto;background:#29343f;color:#fff;border-bottom-right-radius:3px}.agent{background:#fff;border:1px solid #ded9d2;border-bottom-left-radius:3px}.meta{font-size:7.5px;text-transform:uppercase;letter-spacing:.8px;opacity:.62;margin-bottom:4px}
.controls{padding:13px 17px;background:#fff;border-top:1px solid #ebe7e1}.select,.message{width:100%;box-sizing:border-box;border:1px solid #d9d3ca;border-radius:8px;background:#fff;padding:10px;font:12px Arial;color:#26313b}.message{height:72px;resize:vertical;margin-top:7px}.buttons{display:flex;gap:7px;margin-top:7px}.send,.reset{border-radius:8px;height:35px;padding:0 17px;cursor:pointer;font-weight:600}.send{background:#8d2435;color:#fff;border:1px solid #8d2435}.send:hover{background:#711b29}.reset{background:#fff;color:#3b454e;border:1px solid #d9d3ca}
.action{padding:10px 0;border-bottom:1px solid #f0ede8}.action:last-child{border:0}.action-title{font-size:10.5px;font-weight:700}.reason{font-size:8.5px;color:#68717d;line-height:1.42;margin-top:4px}.badge{display:inline-block;padding:3px 7px;border-radius:12px;margin-top:5px;font-size:7px;font-weight:700;letter-spacing:.6px}.ok{background:#e8f2eb;color:#2f6b50}.warn{background:#f7efdf;color:#876528}
.journey{height:188px;margin-top:15px;position:relative;overflow:hidden;background:linear-gradient(135deg,#fff,#faf0ed)}.journey-copy{position:relative;z-index:5;padding:23px 20px;width:175px}.quote{font:italic 600 20px/1.1 Georgia,serif;color:#3a292d}.rule{width:32px;border-top:2px solid #8d2435;margin:12px 0}.small{font-size:7.5px;line-height:1.6;letter-spacing:1.8px;color:#7d8389;font-weight:700}.sun{position:absolute;right:28px;bottom:39px;width:58px;height:58px;border-radius:50%;background:#efd6ae;opacity:.65}.cloud{position:absolute;right:-2px;bottom:18px;width:175px;height:40px;background:#fff;border-radius:50px;opacity:.9}.cloud:before,.cloud:after{content:"";position:absolute;background:#fff;border-radius:50%}.cloud:before{width:70px;height:70px;right:67px;bottom:5px}.cloud:after{width:50px;height:50px;right:18px;bottom:3px}.plane{position:absolute;right:35px;bottom:42px;color:#8d2435;font-size:42px;transform:rotate(-17deg);z-index:6}
.typing{display:flex;align-items:center;gap:5px;width:max-content;padding:10px 13px;background:#fff;border:1px solid #ded9d2;border-radius:11px;border-bottom-left-radius:3px;margin-bottom:10px;font-size:10px;color:#7a838c}
.typing-dot{width:5px;height:5px;border-radius:50%;background:#8d2435;animation:aionosTyping 1.1s infinite ease-in-out}
.typing-dot:nth-child(2){animation-delay:.15s}.typing-dot:nth-child(3){animation-delay:.3s}
@keyframes aionosTyping{0%,60%,100%{opacity:.25;transform:translateY(0)}30%{opacity:1;transform:translateY(-2px)}}
.footer{text-align:center;color:#8b9197;font-size:8px;margin-top:12px}
@media(max-width:1100px){.a-grid,.a-hero{grid-template-columns:1fr}.a-personas{flex-wrap:wrap}}
</style>

<div class="aionos-app" id="aionosApp">
  <div class="a-head">
    <div class="a-brand">
      <div class="a-mark">✦</div><div class="a-name">AIRFLY</div><div class="a-divider"></div>
      <div><div class="a-sub">RESOLUTION DESK</div><div class="a-caption">PEOPLE · PLANS · POSSIBILITIES</div></div>
    </div>
    <div class="a-note">Turning disruptions into smoother journeys. <span></span>✈</div>
  </div>

  <div class="a-hero">
    <div class="a-card a-main">
      <div class="a-eyebrow">Customer-facing resolution agent</div>
      <div class="a-title">Resolve the disruption.<br>Respect the policy.</div>
      <div class="a-desc">Understand the request, apply the supplied policy, take the right action, and know when a human should take over.</div>
    </div>
    <div class="a-card a-mode"><div class="a-eyebrow">Operating mode</div><strong>Policy grounded</strong><small>Closed-world assignment data · transparent simulated execution</small></div>
  </div>

  <div class="a-label">Select customer persona</div>
  <div class="a-personas">
    <button class="persona active" data-persona="Priya Nair">Priya Nair</button>
    <button class="persona" data-persona="Arvind Kulkarni">Arvind Kulkarni</button>
    <button class="persona" data-persona="Meher Kaur">Meher Kaur</button>
  </div>

  <div class="a-grid">
    <div>
      <div class="a-panel" id="infoPanel"></div>
      <div class="a-panel" style="margin-top:15px" id="policyPanel"></div>
    </div>

    <div class="a-panel">
      <div class="chat-head">
        <div><div class="chat-name" id="chatName"></div><div class="pnr" id="chatPnr"></div></div>
        <div class="live"><span class="dot"></span>LIVE SESSION</div>
      </div>
      <div class="chat" id="chat"></div>
      <div class="controls">
        <div class="a-eyebrow">Customer message</div>
        <select class="select" id="customerSelect"></select>
        <textarea class="message" id="message" placeholder="Type the customer's request…"></textarea>
        <div class="buttons"><button class="send" id="send">Send message</button><button class="reset" id="reset">Reset chat</button></div>
      </div>
    </div>

    <div>
      <div class="a-panel" id="actionsPanel"></div>
      <div class="journey a-card">
        <div class="journey-copy"><div class="quote">Every journey<br>has a next chapter.</div><div class="rule"></div><div class="small">WE'RE HERE<br>TO HELP YOU REACH IT.</div></div>
        <div class="sun"></div><div class="cloud"></div><div class="plane">✈</div>
      </div>
    </div>
  </div>
  <div class="footer">AIRFLY Resolution Desk · Customer service prototype</div>
</div>

<script>
(function(){
const CUSTOMERS={
"Priya Nair":{tier:"Gold",pnr:"SK4821X",flight:"SK-204",route:"Delhi → Goa",date:"Wed 23 Sep 2026",time:"18:40",status:"Cancelled",detail:"operational reasons"},
"Arvind Kulkarni":{tier:"Silver",pnr:"TR1190B",flight:"SK-118",route:"Mumbai → Bengaluru",date:"Wed 23 Sep 2026",time:"07:10",status:"Delayed 4h",detail:"new departure 11:10"},
"Meher Kaur":{tier:"Platinum",pnr:"WL7742",flight:"SK-305",route:"Delhi → Hyderabad",date:"Wed 23 Sep 2026",time:"14:00",status:"Delayed 6h",detail:"new departure 20:00"}
};
const SCENARIOS={
"Priya Nair":"I am furious. My flight was cancelled and I want a full cash refund plus a free business class upgrade on my return flight.",
"Arvind Kulkarni":"My flight is delayed by 4 hours and I need a hotel because I am missing an important meeting.",
"Meher Kaur":"I want a full night's hotel stay and I want to move to a different higher-fare flight. The fare difference is ₹2,000."
};
const sessions={};
Object.keys(CUSTOMERS).forEach(n=>sessions[n]={messages:[],actions:[]});
let current="Priya Nair";

function esc(s){return String(s).replace(/[&<>"']/g,m=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#039;"}[m]));}
function hours(c){let m=CUSTOMERS[c].status.match(/(\d+)/);return m?Number(m[1]):0;}

function info(c){
const d=CUSTOMERS[c];
return `<div class="a-pt">Customer & booking</div><div class="a-pb">
<div class="info"><div class="k">Customer</div><div class="v">${esc(c)}</div></div>
<div class="info"><div class="k">Loyalty</div><div class="v">${d.tier}</div></div>
<div class="info"><div class="k">PNR</div><div class="v">${d.pnr}</div></div>
<div class="info"><div class="k">Flight</div><div class="v">${d.flight}</div></div>
<div class="info"><div class="k">Route</div><div class="v">${d.route}</div></div>
<div class="info"><div class="k">Date</div><div class="v">${d.date}</div></div>
<div class="info"><div class="k">Scheduled</div><div class="v">${d.time}</div></div>
<div class="info"><div class="k">Status</div><div class="v"><span class="status">${d.status}</span></div></div>
</div>`;
}
function policy(c){
const d=CUSTOMERS[c],h=hours(c),a=[];
if(d.status==="Cancelled")a.push("Cancellation: free rebooking within 24h OR full refund.");
if(h>5)a.push("Delay >5h: meal + lounge + hotel for delayed hours only.");
else if(h>3)a.push("Delay >3h: meal voucher + lounge access.");
else if(h>0)a.push("Delay <3h: ₹500 meal voucher.");
a.push("Gold/Platinum: priority rebooking, no extra compensation.");
return `<div class="a-pt">Applicable policy</div><div class="a-pb">${a.map(x=>`<div class="policy">${x}</div>`).join("")}</div>`;
}
function renderChat(c){
const box=document.getElementById("chat"),msgs=sessions[c].messages;
if(!msgs.length){box.innerHTML=`<div class="empty"><div><div class="empty-icon">•••</div>Start with a scenario above or type a customer request below.<br>This conversation is saved independently for <b>${esc(c)}</b>.</div></div>`;return;}
box.innerHTML=msgs.map(m=>`<div class="bubble ${m.role==="customer"?"customer":"agent"}"><div class="meta">${m.role==="customer"?"Customer":"Resolution Agent"}</div>${esc(m.text)}</div>`).join("");
box.scrollTop=box.scrollHeight;
}
function renderActions(c){
const a=sessions[c].actions;
let body=a.length?a.slice().reverse().map(x=>`<div class="action"><div class="action-title">${esc(x.action)}</div><span class="badge ${x.status==="EXECUTED"?"ok":"warn"}">${x.status}</span><div class="reason">${esc(x.reason)}</div></div>`).join(""):`<div style="font-size:9.5px;color:#68717d">No actions yet for this persona.</div>`;
document.getElementById("actionsPanel").innerHTML=`<div class="a-pt">Action record · ${esc(c)}</div><div class="a-pb">${body}</div>`;
}
function render(){
const d=CUSTOMERS[current];
document.getElementById("infoPanel").innerHTML=info(current);
document.getElementById("policyPanel").innerHTML=policy(current);
document.getElementById("actionsPanel").innerHTML=actionsPanel(current);
document.getElementById("chatName").textContent=current;
document.getElementById("chatPnr").textContent="PNR "+d.pnr;
renderChat(current);renderActions(current);
document.querySelectorAll(".persona").forEach(b=>b.classList.toggle("active",b.dataset.persona===current));
document.getElementById("customerSelect").value=current;
}
function actionsPanel(c){return `<div class="a-pt">Action record · ${esc(c)}</div><div class="a-pb" id="actionsInner"></div>`}
function addAction(c,action,status,reason){sessions[c].actions.push({action,status,reason});}

function respond(c,text){
const d=CUSTOMERS[c], lower=text.toLowerCase(), h=hours(c), s=sessions[c];
s.messages.push({role:"customer",text});
renderChat(c);

/* Realistic service-desk behaviour: the agent pauses briefly while
   understanding the request and checking the policy. */
const box=document.getElementById("chat");
const typing=document.createElement("div");
typing.className="typing";
typing.innerHTML='<span class="typing-dot"></span><span class="typing-dot"></span><span class="typing-dot"></span><span style="margin-left:3px">Checking policy…</span>';
box.appendChild(typing);
box.scrollTop=box.scrollHeight;

document.getElementById("send").disabled=true;
document.getElementById("send").style.opacity=".55";
document.getElementById("message").disabled=true;

setTimeout(()=>{
  let out=[],matched=false;
  const angry=/furious|angry|unacceptable|ridiculous|worst|legal action|lawyer|sue/.test(lower);
  if(angry)out.push("I understand this has been frustrating, and I’ll help with what I’m authorized to do.");

  if(/refund|money back|cash back/.test(lower)){
   matched=true;
   if(d.status==="Cancelled"){
     out.push("Your full refund request is eligible under the airline-caused cancellation policy. I’ve initiated the refund to the original payment method.");
     addAction(c,"Initiate full refund","EXECUTED","Airline-caused cancellation qualifies for a full refund.");
   } else {
     out.push("I can’t authorize that refund from the supplied policy, so I’m escalating it to a human agent.");
     addAction(c,"Refund review","ESCALATED","The supplied data does not authorize this refund request.");
   }
  }
  if(/rebook|another flight|move me|different flight/.test(lower)){
   matched=true;
   if(d.status==="Cancelled"){
     out.push("Free rebooking is available on the next available flight within 24 hours.");
     addAction(c,"Free rebooking","EXECUTED","Airline-caused cancellation qualifies for free rebooking within 24 hours.");
   } else out.push("The supplied data does not establish automatic rebooking eligibility for this request.");
  }
  if(/business class|upgrade|free upgrade/.test(lower)){
   matched=true;
   out.push("I can’t authorize a free business-class upgrade under the supplied policy. I’m escalating this request to a human agent.");
   addAction(c,"Human review for upgrade","ESCALATED","The supplied policy does not authorize a free business-class upgrade.");
  }
  if(/meal|food|voucher/.test(lower)){
   matched=true;
   if(h>3){out.push("A meal voucher is covered for this delay.");addAction(c,"Issue meal voucher","EXECUTED","Delay is more than 3 hours.");}
   else if(h>0){out.push("A ₹500 meal voucher is covered for this delay.");addAction(c,"Issue ₹500 meal voucher","EXECUTED","Delay is under 3 hours.");}
  }
  if(/lounge/.test(lower)){
   matched=true;
   if(h>3){out.push("Lounge access is covered for this delay.");addAction(c,"Provide lounge access","EXECUTED","Delay is more than 3 hours.");}
   else out.push("Lounge access is not covered by the supplied policy for this delay.");
  }
  if(/hotel|stay|accommodation/.test(lower)){
   matched=true;
   if(h>5){
     out.push("Hotel accommodation is covered for the delayed-hours portion only, not a full-night stay.");
     addAction(c,"Arrange hotel for delayed hours","EXECUTED","Delay is more than 5 hours; policy covers delayed hours only.");
   } else out.push("Hotel accommodation applies only to delays of more than 5 hours.");
  }
  if(/fare difference|₹2000|2000|higher fare|higher-fare/.test(lower)){
   matched=true;
   out.push("The ₹2,000 fare difference requires supervisor approval because waivers above ₹1,500 are outside my authority. I’m escalating it.");
   addAction(c,"Supervisor approval","ESCALATED","Fare difference above ₹1,500 requires supervisor approval; this request is ₹2,000.");
  }
  if(/legal|lawyer|sue|lawsuit|formal complaint/.test(lower)){
   matched=true;
   out.push("I’m escalating the legal/formal complaint request to a human agent.");
   addAction(c,"Human escalation","ESCALATED","Legal action or a formal complaint must be escalated immediately.");
  }
  if(!matched)out.push("I can help with your current disruption. You can ask about a refund, rebooking, meals, lounge access, hotel coverage or another request.");

  s.messages.push({role:"agent",text:out.join(" ")});
  render();
  document.getElementById("send").disabled=false;
  document.getElementById("send").style.opacity="1";
  document.getElementById("message").disabled=false;
  document.getElementById("message").focus();
}, 900);
}

const sel=document.getElementById("customerSelect");
Object.keys(CUSTOMERS).forEach(n=>{let o=document.createElement("option");o.value=n;o.textContent=n;sel.appendChild(o);});
document.querySelectorAll(".persona").forEach(b=>b.addEventListener("click",()=>{
 current=b.dataset.persona;
 document.getElementById("message").value="";
 render();
 document.getElementById("message").value=SCENARIOS[current];
}));
sel.addEventListener("change",()=>{current=sel.value;document.getElementById("message").value="";render();});
document.getElementById("send").addEventListener("click",()=>{
let m=document.getElementById("message").value.trim();if(!m)return;respond(current,m);document.getElementById("message").value="";
});
document.getElementById("message").addEventListener("keydown",(e)=>{
if(e.key==="Enter" && !e.shiftKey){
e.preventDefault();
if(!document.getElementById("send").disabled) document.getElementById("send").click();
}
});
document.getElementById("reset").addEventListener("click",()=>{
sessions[current]={messages:[],actions:[]};document.getElementById("message").value="";render();
});
render();
})();
</script>
"""))


In [ ]:

def show_audit_log(customer=None):
    customer = customer or customer_select.value
    records = current_session(customer)["action_log"]

    if not records:
        print(f"No actions recorded for {customer}.")
        return

    for item in records:
        print(
            f'[{item["timestamp"]}] {item["customer"]} | '
            f'{item["action"]} | {item["status"]} | {item["reason"]}'
        )
